**Part 7 · RAG Mastery** · step 36 of 37
[← 35 Lab: search from scratch](35_lab_search_from_scratch.ipynb) · [🏠 Roadmap](../README.md) · [37 Lab: agentic RAG →](37_lab_agentic_rag_langgraph.ipynb)

# 36 · Lab: Hybrid Search + Cross-Encoder Re-ranking, Measured Stage by Stage

**No API key needed.** It downloads two small open models (~100 MB total, first run only) and runs on CPU.

**What you'll build and measure:**
1. **BM25** (keyword) retriever
2. **Dense** retriever: `all-MiniLM-L6-v2` bi-encoder + **FAISS**
3. **Hybrid**: BM25 + dense fused with **RRF**
4. **Re-ranking** with a **cross-encoder** (`ms-marco-MiniLM-L-6-v2`)
5. An eval table showing **which stage fixes which question**
6. Latency per stage, exact vs **HNSW** search, and a reranker **score threshold** for "I don't know"

Theory: [27](27_embeddings_and_vector_search.md), [28](28_bm25_and_elasticsearch.md), [29](29_hybrid_search_and_reranking.md), [33](33_evaluation_and_production.md).

In [ ]:
import math
import re
import time
from collections import Counter, defaultdict

import numpy as np
from sentence_transformers import CrossEncoder, SentenceTransformer
import faiss   # import AFTER sentence_transformers: on macOS the other order can segfault (OpenMP clash)

faiss.omp_set_num_threads(1)   # FAISS and PyTorch both use OpenMP; sharing threads can crash on macOS

from nimbus_corpus import DOCS, EVAL

ids = [d["id"] for d in DOCS]
texts = [f"{d['title']}. {d['text']}" for d in DOCS]      # title + body = a tiny "contextual header"
text_by_id = dict(zip(ids, texts))
print(len(DOCS), "docs |", len(EVAL), "questions")

## 1. BM25 retriever (compact version of lab 35)

In [ ]:
STOP = {"a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "is", "are", "be", "it", "my", "i", "do", "does",
        "what", "how", "can", "with", "per", "at", "by", "from", "if", "your", "you", "their"}

def stem(word: str) -> str:
    for suffix in ("ing", "ed", "es", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            word = word[: -len(suffix)]
            return word[:-1] if suffix == "ing" and word[-1] == word[-2] else word
    return word

def analyze(text: str) -> list[str]:
    toks = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())     # keeps E-4021 / 88-1032 as one token
    return [stem(t) for t in toks if t not in STOP]

class BM25:
    def __init__(self, texts: list[str], k1: float = 1.2, b: float = 0.75):
        self.k1, self.b = k1, b
        self.docs = [Counter(analyze(t)) for t in texts]
        self.lens = [sum(d.values()) for d in self.docs]
        self.avgdl = sum(self.lens) / len(self.lens)
        df = Counter(term for d in self.docs for term in d)
        self.idf = {t: math.log(1 + (len(texts) - n + 0.5) / (n + 0.5)) for t, n in df.items()}

    def scores(self, query: str) -> np.ndarray:
        out = np.zeros(len(self.docs))
        for t in analyze(query):
            for i, d in enumerate(self.docs):
                if t in d:
                    tf = d[t]
                    out[i] += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.lens[i] / self.avgdl))
        return out

bm25 = BM25(texts)

def bm25_search(q: str, k: int = 20) -> list[str]:
    s = bm25.scores(q)
    return [ids[i] for i in np.argsort(-s)[:k] if s[i] > 0]

print(bm25_search("VPN password")[:3])

## 2. Dense retriever: bi-encoder + FAISS

Documents are embedded **once** (offline). At query time we embed only the query.
Vectors are normalised, so inner product = cosine similarity.

In [ ]:
def load_on_cpu(model):
    """Copy weights into normal RAM. Workaround: with some torch/transformers versions on macOS, memory-mapped
    weights make CPU inference return NaN. Harmless everywhere else."""
    for p in model.parameters():
        p.data = p.data.clone()
    return model

embedder = load_on_cpu(SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu"))   # 384-dim

doc_vecs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False).astype("float32")
flat_index = faiss.IndexFlatIP(doc_vecs.shape[1])      # exact search (fine for small data)
flat_index.add(doc_vecs)

def dense_search(q: str, k: int = 20) -> list[str]:
    qv = embedder.encode([q], normalize_embeddings=True).astype("float32")
    _, idx = flat_index.search(qv, k)
    return [ids[i] for i in idx[0] if i != -1]

print("vectors:", doc_vecs.shape)
print(dense_search("How many vacation days do I get?")[:3])

## 3. Hybrid: Reciprocal Rank Fusion

In [ ]:
def rrf(lists: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = defaultdict(float)
    for lst in lists:
        for rank, d in enumerate(lst, start=1):
            scores[d] += 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_search(q: str, k: int = 20) -> list[str]:
    return rrf([bm25_search(q, k), dense_search(q, k)])[:k]

print(hybrid_search("E-4210")[:3])

## 4. Re-ranking with a cross-encoder

The cross-encoder reads **query and document together**, so it's much more precise, but it must run once per
pair. That's why it only reranks the **top 20** from hybrid search, never the whole corpus.

Its scores are **logits** (unbounded): positive = relevant, very negative = irrelevant.

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
load_on_cpu(reranker.model)

def rerank(q: str, candidates: list[str]) -> list[tuple[str, float]]:
    scores = reranker.predict([(q, text_by_id[d]) for d in candidates])
    return sorted(zip(candidates, map(float, scores)), key=lambda x: x[1], reverse=True)

def full_search(q: str, k: int = 20) -> list[str]:
    return [d for d, _ in rerank(q, hybrid_search(q, k))]

for d, s in rerank("what to tell a customer about E-4201", hybrid_search("what to tell a customer about E-4201"))[:3]:
    print(f"{s:7.2f}  {d}")

## 5. Evaluate every stage on the same labelled questions

In [ ]:
def rank_of(ranked: list[str], relevant: set[str]) -> int | None:
    return next((r for r, d in enumerate(ranked, start=1) if d in relevant), None)

STAGES = {"bm25": bm25_search, "dense": dense_search, "hybrid": hybrid_search, "+rerank": full_search}
ranks = {name: [rank_of(fn(ex["q"]), set(ex["relevant"])) for ex in EVAL] for name, fn in STAGES.items()}

print(f"{'kind':<11} {'question':<52}" + "".join(f"{n:>9}" for n in STAGES))
for i, ex in enumerate(EVAL):
    cells = "".join(f"{(str(ranks[n][i]) if ranks[n][i] else '-'):>9}" for n in STAGES)
    print(f"{ex['kind']:<11} {ex['q'][:50]:<52}{cells}")

In [ ]:
def summary(rs: list[int | None]) -> dict:
    return {"hit@1": round(float(np.mean([r == 1 for r in rs])), 3),
            "recall@5": round(float(np.mean([r is not None and r <= 5 for r in rs])), 3),
            "mrr": round(float(np.mean([1 / r if r else 0 for r in rs])), 3)}

for name in STAGES:
    print(f"{name:<8}", summary(ranks[name]))

**Read the table carefully, this is the whole lesson:**
- **BM25** fails on *synonym* questions ("suspicious email" vs "phishing", "maternity" vs "parental").
- **Dense** fails on *look-alike codes* (E-4120 vs E-4021, E-4201 vs E-4210: almost the same vector!) and on some
  synonyms where a related-but-wrong doc is closer ("holidays" → the public holiday calendar).
- They fail on **different** questions → **hybrid** beats both.
- The **cross-encoder** reads query + doc together and fixes the remaining ranking errors.

(Our corpus is tiny, so numbers are high. On real data the pattern is the same, the gaps are just bigger.)

## 6. Latency per stage (the price of each improvement)

In [ ]:
def timed(fn, q, n=5):
    t = time.perf_counter()
    for _ in range(n):
        fn(q)
    return (time.perf_counter() - t) / n * 1000

q = "Can I get my home wifi bill refunded?"
cands = hybrid_search(q)
print(f"bm25      {timed(bm25_search, q):7.2f} ms")
print(f"dense     {timed(dense_search, q):7.1f} ms   (mostly embedding the query)")
print(f"hybrid    {timed(hybrid_search, q):7.1f} ms")
print(f"rerank 20 {timed(lambda x: rerank(x, cands), q):7.1f} ms   (20 cross-encoder passes)")

Reranking is the most expensive retrieval step and grows linearly with the number of candidates.
That's the **funnel**: cheap retrievers over everything, the expensive reranker only over the top 20–100.

## 7. Exact vs approximate (HNSW) search

With 27 docs exact search is instant. With millions you'd use HNSW. Let's simulate 50k random vectors and compare
speed and **recall vs exact search** for different `efSearch` values.

In [ ]:
rng = np.random.default_rng(0)
centers = rng.standard_normal((500, 384))                         # real embeddings form topic clusters
big = (centers[rng.integers(0, 500, 50_000)] + 0.6 * rng.standard_normal((50_000, 384))).astype("float32")
big /= np.linalg.norm(big, axis=1, keepdims=True)
queries = big[rng.choice(len(big), 200, replace=False)] + 0.02 * rng.standard_normal((200, 384)).astype("float32")
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

exact = faiss.IndexFlatIP(384); exact.add(big)
t = time.perf_counter(); _, true_ids = exact.search(queries, 10); t_exact = time.perf_counter() - t

hnsw = faiss.IndexHNSWFlat(384, 32, faiss.METRIC_INNER_PRODUCT)   # M = 32 links per node
hnsw.hnsw.efConstruction = 80
hnsw.add(big)
for ef in (4, 16, 64, 256):
    hnsw.hnsw.efSearch = ef
    t = time.perf_counter(); _, got = hnsw.search(queries, 10); t_h = time.perf_counter() - t
    recall = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(got, true_ids)])
    print(f"efSearch={ef:<4} recall@10={recall:.3f}  time={t_h * 1000:6.1f} ms   (exact: {t_exact * 1000:.1f} ms)")

Bigger `efSearch` → higher recall, slower. The same knob is `num_candidates` in Elasticsearch kNN.
Exact timings depend on your machine; the trade-off is what matters. At 50k vectors exact search is still OK,
at 50M it isn't, while HNSW query time grows only roughly logarithmically.

## 8. Abstain when nothing is relevant ("I don't know")

The reranker score is a good **confidence signal**. If even the best chunk scores low, don't let the LLM guess.

In [ ]:
THRESHOLD = 0.0   # tune on your eval set: look at best-score distributions for answerable vs unanswerable questions

for q in ["What does E-4021 mean?", "How many vacation days do I get per year?",
          "What is the office dress code?", "Who won the cricket world cup?"]:
    best_id, best = rerank(q, hybrid_search(q))[0]
    verdict = f"answer from {best_id}" if best > THRESHOLD else "→ say \"I don't know\""
    print(f"{best:7.2f}  {q:<45} {verdict}")

## 9. Exercise ideas
1. Add a question your system gets wrong, then fix it (synonym? metadata filter? better doc text?).
2. Try `BAAI/bge-small-en-v1.5` as the embedder (remember: same model for docs and queries).
3. Add a `department` filter: only search HR docs when the question is about leave.
4. Remove titles from `texts` (body only) and watch metrics drop: titles act as **contextual headers**.
5. Weight the RRF lists (e.g. 0.7 dense / 0.3 BM25) and compare.

## Revision notes
- BM25 and dense fail on **different** questions (synonyms vs look-alike codes) → **hybrid with RRF** beats both.
- Cross-encoder reranking fixes the final order; it's the slowest retrieval step, so only rerank the top 20–100.
- Always evaluate **per stage** with the same labelled set (hit@1, recall@k, MRR) and look at **which questions** changed.
- HNSW `efSearch` / ES `num_candidates`: recall vs speed knob.
- Reranker score threshold = cheap, effective **abstain** signal against hallucination.

---
✅ **Done with 36.** Next: [37 Lab: agentic RAG with LangGraph →](37_lab_agentic_rag_langgraph.ipynb)